In [1]:
# Get descriptive statistics.

import zipfile
from PIL import Image
import io
import numpy as np

zip_path = "FaceForensics.zip"

res_real = []
res_fake = []

count = 0

with zipfile.ZipFile(zip_path, 'r') as z:
    # Only include image files
    files = [
        f for f in z.namelist()
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print(f"Found {len(files)} image files inside ZIP")

    for fname in files:
        count += 1
        if count % 10000 == 0:
            print(f"Processed {count} images...")

        lower = fname.lower()

        # Skip anything not inside train/val/test
        if not ("/train/" in lower or "/val/" in lower or "/test/" in lower):
            continue

        # Determine class
        # Real images are ONLY in folders literally named "Real"
        if "/real/" in lower:
            label = "real"
        else:
            # Any other subfolder inside a video folder is fake
            label = "fake"

        # Load image
        try:
            with z.open(fname) as f:
                img = Image.open(io.BytesIO(f.read()))
                w, h = img.size
        except:
            print("Warning: could not open", fname)
            continue

        # Store resolution
        if label == "real":
            res_real.append((w, h))
        else:
            res_fake.append((w, h))

print("Done!\n")

# Convert to arrays
real_arr = np.array(res_real)
fake_arr = np.array(res_fake)

print("=== REAL ===")
print("Count:", len(real_arr))
print("Min:", real_arr.min(axis=0))
print("Max:", real_arr.max(axis=0))
print("Mean:", real_arr.mean(axis=0))

print("\n=== FAKE ===")
print("Count:", len(fake_arr))
print("Min:", fake_arr.min(axis=0))
print("Max:", fake_arr.max(axis=0))
print("Mean:", fake_arr.mean(axis=0))

Found 182246 image files inside ZIP
Processed 10000 images...
Processed 20000 images...
Processed 30000 images...
Processed 40000 images...
Processed 50000 images...
Processed 60000 images...
Processed 70000 images...
Processed 80000 images...
Processed 90000 images...
Processed 100000 images...
Processed 110000 images...
Processed 120000 images...
Processed 130000 images...
Processed 140000 images...
Processed 150000 images...
Processed 160000 images...
Processed 170000 images...
Processed 180000 images...
Done!

=== REAL ===
Count: 26444
Min: [224 224]
Max: [224 224]
Mean: [224. 224.]

=== FAKE ===
Count: 155802
Min: [224 224]
Max: [224 224]
Mean: [224. 224.]


In [1]:
# Unify file types and image resolutions.

import zipfile
from PIL import Image
import io
import os

zip_path = "FaceForensics.zip"
output_dir = "dataset_3_cleaned"
target_size = (256, 256)

os.makedirs(output_dir, exist_ok=True)

count = 0

with zipfile.ZipFile(zip_path, 'r') as z:
    # Include only image files
    files = [
        f for f in z.namelist()
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    print(f"Found {len(files)} image files inside ZIP")

    for fname in files:
        lower = fname.lower()

        # Only process train/val/test folders
        if not ("/train/" in lower or "/val/" in lower or "/test/" in lower):
            continue

        count += 1
        if count % 1000 == 0:
            print(f"Processed {count} images...")

        # Determine class
        # Real images are ONLY in folders named "Real"
        if "/real/" in lower:
            cls = "real"
        else:
            cls = "fake"

        class_dir = os.path.join(output_dir, cls)
        os.makedirs(class_dir, exist_ok=True)

        # Load image
        try:
            with z.open(fname) as f:
                img = Image.open(io.BytesIO(f.read())).convert("RGB")
        except:
            print("Warning: could not open", fname)
            continue

        # Resize
        img = img.resize(target_size, Image.LANCZOS)

        # Build a unique base name from the full path inside the ZIP
        rel_path = fname

        # Strip leading root folder (e.g., "FaceForensics/")
        if "/" in rel_path:
            rel_path = rel_path.split("/", 1)[1]

        # Remove extension
        base_no_ext = os.path.splitext(rel_path)[0]

        # Replace slashes with underscores to make a safe filename
        safe_base = base_no_ext.replace("/", "_").replace("\\", "_")

        suffix = "_real" if cls == "real" else "_fake"
        out_name = safe_base + suffix + ".jpg"
        out_path = os.path.join(class_dir, out_name)

        # Save as uniform JPG
        img.save(out_path, "JPEG", quality=95, subsampling=2)

print("Done! All images converted and resized.")

Found 182246 image files inside ZIP
Processed 1000 images...
Processed 2000 images...
Processed 3000 images...
Processed 4000 images...
Processed 5000 images...
Processed 6000 images...
Processed 7000 images...
Processed 8000 images...
Processed 9000 images...
Processed 10000 images...
Processed 11000 images...
Processed 12000 images...
Processed 13000 images...
Processed 14000 images...
Processed 15000 images...
Processed 16000 images...
Processed 17000 images...
Processed 18000 images...
Processed 19000 images...
Processed 20000 images...
Processed 21000 images...
Processed 22000 images...
Processed 23000 images...
Processed 24000 images...
Processed 25000 images...
Processed 26000 images...
Processed 27000 images...
Processed 28000 images...
Processed 29000 images...
Processed 30000 images...
Processed 31000 images...
Processed 32000 images...
Processed 33000 images...
Processed 34000 images...
Processed 35000 images...
Processed 36000 images...
Processed 37000 images...
Processed 3

In [2]:
# Validate updated dataset.

import os
from PIL import Image
import numpy as np

cleaned_path = "dataset_3_cleaned"

real_dir = os.path.join(cleaned_path, "real")
fake_dir = os.path.join(cleaned_path, "fake")

res_real = []
res_fake = []

# --- REAL IMAGES ---
real_files = [f for f in os.listdir(real_dir) if f.lower().endswith(".jpg")]

for i, fname in enumerate(real_files):
    if i % 10000 == 0:
        print(f"Processed {i} real images...")

    img_path = os.path.join(real_dir, fname)
    img = Image.open(img_path)
    w, h = img.size
    res_real.append((w, h))

# --- FAKE IMAGES ---
fake_files = [f for f in os.listdir(fake_dir) if f.lower().endswith(".jpg")]

for i, fname in enumerate(fake_files):
    if i % 10000 == 0:
        print(f"Processed {i} fake images...")

    img_path = os.path.join(fake_dir, fname)
    img = Image.open(img_path)
    w, h = img.size
    res_fake.append((w, h))

print("Done!")

# Convert to arrays for stats
real_arr = np.array(res_real)
fake_arr = np.array(res_fake)

print("=== REAL (cleaned JPG) ===")
print("Count:", len(real_arr))
print("Min:", real_arr.min(axis=0))
print("Max:", real_arr.max(axis=0))
print("Mean:", real_arr.mean(axis=0))

print("\n=== FAKE (cleaned JPG) ===")
print("Count:", len(fake_arr))
print("Min:", fake_arr.min(axis=0))
print("Max:", fake_arr.max(axis=0))
print("Mean:", fake_arr.mean(axis=0))

Processed 0 real images...
Processed 10000 real images...
Processed 20000 real images...
Processed 0 fake images...
Processed 10000 fake images...
Processed 20000 fake images...
Processed 30000 fake images...
Processed 40000 fake images...
Processed 50000 fake images...
Processed 60000 fake images...
Processed 70000 fake images...
Processed 80000 fake images...
Processed 90000 fake images...
Processed 100000 fake images...
Processed 110000 fake images...
Processed 120000 fake images...
Processed 130000 fake images...
Processed 140000 fake images...
Processed 150000 fake images...
Done!
=== REAL (cleaned JPG) ===
Count: 26444
Min: [256 256]
Max: [256 256]
Mean: [256. 256.]

=== FAKE (cleaned JPG) ===
Count: 155802
Min: [256 256]
Max: [256 256]
Mean: [256. 256.]


In [3]:
# Color comparison between real and fake images.

import os
import numpy as np
from PIL import Image

cleaned_path = "dataset_3_cleaned"
real_dir = os.path.join(cleaned_path, "real")
fake_dir = os.path.join(cleaned_path, "fake")

def compute_color_stats(folder):
    means = []
    stds = []

    files = [f for f in os.listdir(folder) if f.lower().endswith(".jpg")]

    for i, fname in enumerate(files):
        if i % 10000 == 0:
            print(f"Processed {i} images in {folder}...")

        img = Image.open(os.path.join(folder, fname)).convert("RGB")
        arr = np.array(img) / 255.0  # normalize to [0,1]

        means.append(arr.mean(axis=(0,1)))
        stds.append(arr.std(axis=(0,1)))

    means = np.array(means)
    stds = np.array(stds)

    return means.mean(axis=0), stds.mean(axis=0)

real_mean, real_std = compute_color_stats(real_dir)
fake_mean, fake_std = compute_color_stats(fake_dir)

print("\n=== REAL ===")
print("Mean RGB:", real_mean)
print("Std RGB:", real_std)

print("\n=== FAKE ===")
print("Mean RGB:", fake_mean)
print("Std RGB:", fake_std)

Processed 0 images in dataset_3_cleaned\real...
Processed 10000 images in dataset_3_cleaned\real...
Processed 20000 images in dataset_3_cleaned\real...
Processed 0 images in dataset_3_cleaned\fake...
Processed 10000 images in dataset_3_cleaned\fake...
Processed 20000 images in dataset_3_cleaned\fake...
Processed 30000 images in dataset_3_cleaned\fake...
Processed 40000 images in dataset_3_cleaned\fake...
Processed 50000 images in dataset_3_cleaned\fake...
Processed 60000 images in dataset_3_cleaned\fake...
Processed 70000 images in dataset_3_cleaned\fake...
Processed 80000 images in dataset_3_cleaned\fake...
Processed 90000 images in dataset_3_cleaned\fake...
Processed 100000 images in dataset_3_cleaned\fake...
Processed 110000 images in dataset_3_cleaned\fake...
Processed 120000 images in dataset_3_cleaned\fake...
Processed 130000 images in dataset_3_cleaned\fake...
Processed 140000 images in dataset_3_cleaned\fake...
Processed 150000 images in dataset_3_cleaned\fake...

=== REAL ===
